[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/19.%20The%20SLAP%20for%20Reefer%20Containers%20Problem/P19-Tier-8_executed.ipynb)

---


# 19. The SLAP for Reefer Containers Problem
## Tier 8: Value-Aligned and Ethical Framework

### Purpose and scope

This notebook demonstrates a value-aligned framework for assigning reefer containers to powered storage locations. A candidate assignment is built greedily, stakeholder utilities and ethical compliance are evaluated, and the best-scoring candidate found during a fixed number of iterations is returned. The example is intended to show how stakeholder priorities and ethical considerations can be incorporated into an assignment decision; it is not a validated operational system or a formal implementation of Constitutional AI.

**Inputs:** container energy requirements, temperature requirements and cargo priorities; location capacities, temperature ranges and operating characteristics; stakeholder utility weights and ethical-principle weights.

**Outputs:** a container-to-location assignment, modeled system outcomes, stakeholder utilities, an ethical-compliance score, detected violations and a report with visual summaries.

### Model assumptions

- Each container is assigned to at most one location, and each location has a shared power-capacity limit. Containers may remain unassigned if no location is feasible when they are considered.
- A location can serve a container when their temperature ranges overlap: the location's operating range must intersect the container's permitted range, defined by its requested temperature and tolerance.
- Containers are considered in descending order of urgency multiplied by cargo value. For each container, the greedy rule selects the feasible location with the highest local score.
- The 100 iterations repeat this deterministic greedy construction. They do not explore different candidates, so the best-of-iterations loop records the same assignment when the input data are unchanged.
- Stakeholder constraints and veto powers are recorded as data but are not enforced by the assignment procedure. Principles marked as hard constraints are also not implemented as feasibility constraints: detected violations instead receive the model's stated score penalty. These distinctions matter when interpreting the results.
- Cost, emissions, safety, fairness and community effects are simplified indicators. The example does not model electrical-network behavior, time-varying demand, container relocation, uncertainty, or a regulatory compliance assessment.

### Solution sequence

1. Define stakeholders, their priorities and utility preferences, and the ethical principles to be evaluated.
2. Define reefer containers and candidate locations.
3. Build a greedy feasible assignment and calculate its outcomes.
4. Aggregate stakeholder utility and weighted ethical compliance, apply violation penalties, and retain the highest-scoring candidate.
5. Report the assignment and inspect stakeholder, ethical and operational indicators.

### Mathematical representation

Let $S$ be the set of stakeholders, $P$ the set of ethical principles, $C$ the set of containers, and $L$ the set of locations. Let $a(c)\in L$ denote the location assigned to container $c$; an unassigned container has no value of $a(c)$. The greedy construction considers containers in priority order and assigns each to the highest-scoring feasible location with sufficient remaining power capacity. Feasibility requires both power capacity and temperature-range overlap.

For stakeholder $s$, the model combines normalized objective outcomes $x_j$ using stakeholder weights $w_{sj}$ and a utility transformation $\phi_s$:

$$u_s=\sum_j w_{sj}\,\phi_s(x_j).$$

The transformation is linear, exponential, or logarithmic according to the stakeholder's utility-function type. Stakeholder utilities are aggregated using priorities $p_s$. Ethical compliance is a weighted mean of principle-level scores $q_k$, with principle weights $\alpha_k$. The candidate score is

$$F=(1-\lambda)\sum_{s\in S}p_su_s+\lambda\frac{\sum_{k\in P}\alpha_kq_k}{\sum_{k\in P}\alpha_k}-50|V|,$$

where $\lambda=0.3$ is the ethical weight and $V$ is the list of detected violations. The model retains the candidate with the largest $F$. These are weighted scores and penalties, not hard ethical feasibility constraints.

The assignment rule is greedy rather than an exact global optimization: a location is selected using its local score when each container is processed, and that choice reduces the location's remaining capacity. If there are $|C|$ containers and $|L|$ locations, one construction considers at most $|C||L|$ container-location pairs; the configured 100 iterations repeat that construction. Because no randomization or other candidate variation is used, the iterations do not increase the search coverage.

### Interpreting the results

A higher candidate score means a larger value under this example's chosen scaling, weights and penalties; it is not a universal measure of ethical quality. Utility scores are based on normalized indicators, while the fixed penalty of 50 per detected violation is much larger than the typical weighted utility and compliance terms. Reported satisfaction is a bounded representation of modeled utility, not a survey of real stakeholders. The example data do not support claims about a larger fleet, measured emissions reductions, or real-world stakeholder satisfaction.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple
import math
import time

import matplotlib.pyplot as plt
import numpy as np

print("Packages imported successfully for the value-aligned framework.")

In [ ]:
@dataclass
class Stakeholder:
    """Stakeholder preferences and governance metadata."""

    id: int
    name: str
    stakeholder_type: str
    utilities: Dict[str, float]
    constraints: Dict[str, Tuple[float, float]]
    priority: float
    veto_power: bool

    def __post_init__(self):
        self.satisfaction = 0.5
        self.utility_history = []
        self.veto_count = 0
        self.participation_rate = 1.0


@dataclass
class EthicalPrinciple:
    """A weighted principle used to score modeled compliance."""

    id: int
    name: str
    principle_type: str
    description: str
    weight: float
    hard_constraint: bool
    violation_penalty: float

    def __post_init__(self):
        self.compliance_score = 1.0
        self.violation_history = []


@dataclass
class UtilityFunction:
    """Utility transformation and parameters for one stakeholder."""

    stakeholder_id: int
    objectives: Dict[str, float]
    utility_function_type: str
    risk_aversion: float
    time_preference: float

    def __post_init__(self):
        self.current_utility = 0.0
        self.utility_trend = []


@dataclass
class EthicalDecision:
    """Decision record with fields for explanation and audit data."""

    id: int
    decision_type: str
    alternatives: List[Dict[str, Any]]
    chosen_alternative: Dict[str, Any]
    ethical_score: float
    stakeholder_impacts: Dict[int, float]
    rationale: str
    ethical_violations: List[str]
    transparency_score: float

    def __post_init__(self):
        self.audit_trail = []
        self.stakeholder_feedback = {}


@dataclass
class ConstitutionalAI:
    """Container for principles and decision records in this example."""

    def __init__(self):
        self.constitution = []
        self.stakeholders = []
        self.decision_history = []
        self.compliance_metrics = defaultdict(float)
        self.ethical_audit_log = []


print("Stakeholder, principle, utility, decision, and governance classes defined.")

In [ ]:
# Stakeholder weights define preferences over normalized benefit indicators.
stakeholders = [
    Stakeholder(
        id=1, name="Terminal Operator", stakeholder_type="operator",
        utilities={"revenue": 0.4, "efficiency": 0.3, "safety": 0.2, "reputation": 0.1},
        constraints={"revenue": (1000, 10000), "efficiency": (0.8, 1.0), "safety": (0.9, 1.0)},
        priority=0.8, veto_power=True
    ),
    Stakeholder(
        id=2, name="Shipping Line Alliance", stakeholder_type="customer",
        utilities={"cost": 0.5, "reliability": 0.3, "service_quality": 0.2},
        constraints={"cost": (50, 500), "reliability": (0.95, 1.0)},
        priority=0.9, veto_power=True
    ),
    Stakeholder(
        id=3, name="Local Community", stakeholder_type="community",
        utilities={"environment": 0.4, "employment": 0.3, "noise_reduction": 0.3},
        constraints={"emissions": (0, 100), "noise_level": (0, 70)},
        priority=0.7, veto_power=False
    ),
    Stakeholder(
        id=4, name="Environmental Agency", stakeholder_type="regulator",
        utilities={"compliance": 0.5, "sustainability": 0.3, "enforcement": 0.2},
        constraints={"compliance_rate": (0.95, 1.0), "emissions": (0, 50)},
        priority=0.9, veto_power=True
    ),
    Stakeholder(
        id=5, name="Port Authority", stakeholder_type="operator",
        utilities={"throughput": 0.4, "safety": 0.3, "revenue": 0.3},
        constraints={"throughput": (1000, 5000), "safety_incidents": (0, 5)},
        priority=0.8, veto_power=False
    )
]

ethical_principles = [
    EthicalPrinciple(
        id=1, name="Environmental Sustainability", principle_type="environmental",
        description="Reduce modeled emissions.", weight=0.30,
        hard_constraint=False, violation_penalty=100.0
    ),
    EthicalPrinciple(
        id=2, name="Fair Competition", principle_type="economic",
        description="Promote balanced use of available locations.", weight=0.20,
        hard_constraint=True, violation_penalty=200.0
    ),
    EthicalPrinciple(
        id=3, name="Safety First", principle_type="safety",
        description="Prefer assignments with backup power.", weight=0.40,
        hard_constraint=True, violation_penalty=500.0
    ),
    EthicalPrinciple(
        id=4, name="Community Welfare", principle_type="social",
        description="Account for the modeled noise-related community indicator.", weight=0.25,
        hard_constraint=False, violation_penalty=150.0
    ),
    EthicalPrinciple(
        id=5, name="Transparency and Accountability", principle_type="transparency",
        description="Expose the modeled decision outcomes and scoring components.", weight=0.15,
        hard_constraint=False, violation_penalty=50.0
    )
]

utility_functions = [
    UtilityFunction(1, {"revenue": 0.4, "efficiency": 0.3, "safety": 0.2, "reputation": 0.1}, "linear", 0.3, 0.05),
    UtilityFunction(2, {"cost": 0.5, "reliability": 0.3, "service_quality": 0.2}, "exponential", 0.5, 0.1),
    UtilityFunction(3, {"environment": 0.4, "employment": 0.3, "noise_reduction": 0.3}, "logarithmic", 0.2, 0.02),
    UtilityFunction(4, {"compliance": 0.5, "sustainability": 0.3, "enforcement": 0.2}, "linear", 0.8, 0.03),
    UtilityFunction(5, {"throughput": 0.4, "safety": 0.3, "revenue": 0.3}, "exponential", 0.4, 0.06)
]

print("Value-aligned framework initialized:")
print(f"  Stakeholders: {len(stakeholders)}")
print(f"  Ethical principles: {len(ethical_principles)}")
print(f"  Utility functions: {len(utility_functions)}")

print("\nStakeholder overview:")
for stakeholder in stakeholders:
    veto_status = "Yes" if stakeholder.veto_power else "No"
    print(
        f"  {stakeholder.name} ({stakeholder.stakeholder_type}): "
        f"priority={stakeholder.priority:.1f}, veto power={veto_status}"
    )

print("\nEthical principles:")
for principle in ethical_principles:
    constraint_label = "Marked hard" if principle.hard_constraint else "Soft-scored"
    print(f"  {principle.name}: weight={principle.weight:.2f}, {constraint_label}")

In [ ]:
class ValueAlignedOptimization:
    """Greedy assignment with stakeholder utility and ethical scoring."""

    def __init__(
        self,
        stakeholders: List[Stakeholder],
        ethical_principles: List[EthicalPrinciple],
        utility_functions: List[UtilityFunction]
    ):
        self.stakeholders = stakeholders
        self.ethical_principles = ethical_principles
        self.utility_functions = utility_functions

        self.constitutional_ai = ConstitutionalAI()
        self.constitutional_ai.constitution = ethical_principles
        self.constitutional_ai.stakeholders = stakeholders

        self.optimization_iterations = 100
        self.convergence_threshold = 0.001
        self.ethical_weight = 0.3

        self.optimization_history = []
        self.ethical_compliance_history = []
        self.stakeholder_satisfaction_history = []

    def calculate_stakeholder_utility(
        self,
        stakeholder_id: int,
        decision_outcomes: Dict[str, float]
    ) -> float:
        """Transform and aggregate the stakeholder's available benefit indicators."""
        stakeholder = next(s for s in self.stakeholders if s.id == stakeholder_id)
        utility_function = next(
            uf for uf in self.utility_functions
            if uf.stakeholder_id == stakeholder_id
        )

        total_utility = 0.0
        for objective, weight in stakeholder.utilities.items():
            if objective not in decision_outcomes:
                continue

            outcome_value = float(np.clip(decision_outcomes[objective], 0.0, 1.0))
            if utility_function.utility_function_type == "exponential":
                objective_utility = 1.0 - np.exp(
                    -utility_function.risk_aversion * outcome_value
                )
            elif utility_function.utility_function_type == "logarithmic":
                objective_utility = np.log1p(outcome_value)
            else:
                objective_utility = outcome_value

            total_utility += weight * objective_utility

        return float(total_utility)

    def check_ethical_compliance(
        self,
        decision_outcomes: Dict[str, float]
    ) -> Tuple[float, List[str]]:
        """Score each principle and record threshold violations once per principle."""
        principle_scores = {}
        violations = []

        for principle in self.ethical_principles:
            if principle.principle_type == "environmental":
                score = decision_outcomes["sustainability"]
                if decision_outcomes["emissions"] > 100:
                    violations.append(
                        f"Environmental emissions exceed the modeled limit: "
                        f"{decision_outcomes['emissions']:.1f} units"
                    )
            elif principle.name == "Fair Competition":
                score = decision_outcomes["fairness_score"]
                if score < 0.8:
                    violations.append(f"Fairness score below threshold: {score:.2f}")
            elif principle.principle_type == "safety":
                score = decision_outcomes["safety_score"]
                if score < 0.95:
                    violations.append(f"Safety score below threshold: {score:.2f}")
            elif principle.principle_type == "social":
                score = decision_outcomes["community_welfare"]
                if score < 0.8:
                    violations.append(
                        f"Community welfare score below threshold: {score:.2f}"
                    )
            else:
                score = decision_outcomes["transparency_score"]

            score = float(np.clip(score, 0.0, 1.0))
            principle.compliance_score = score
            principle_scores[principle.name] = score

        total_weight = sum(principle.weight for principle in self.ethical_principles)
        if total_weight <= 0:
            raise ValueError("The total ethical-principle weight must be positive.")

        total_compliance = sum(
            principle.weight * principle_scores[principle.name]
            for principle in self.ethical_principles
        ) / total_weight
        self.latest_principle_scores = principle_scores

        return total_compliance, violations

    def optimize_assignment(
        self,
        containers: List[Dict[str, Any]],
        locations: List[Dict[str, Any]]
    ) -> Dict[str, Any]:
        """Run the configured greedy constructions and retain the best score."""
        self.optimization_history.clear()
        best_solution = None
        best_score = -np.inf

        for iteration in range(self.optimization_iterations):
            candidate_solution = self._generate_candidate_solution(containers, locations)
            decision_outcomes = self._calculate_outcomes(
                candidate_solution, containers, locations
            )

            stakeholder_utilities = {}
            weighted_stakeholder_utility = 0.0
            for stakeholder in self.stakeholders:
                utility = self.calculate_stakeholder_utility(
                    stakeholder.id, decision_outcomes
                )
                stakeholder_utilities[stakeholder.id] = utility
                weighted_stakeholder_utility += stakeholder.priority * utility

            ethical_compliance, violations = self.check_ethical_compliance(
                decision_outcomes
            )
            total_score = (
                (1 - self.ethical_weight) * weighted_stakeholder_utility
                + self.ethical_weight * ethical_compliance
                - 50.0 * len(violations)
            )

            if total_score > best_score:
                best_score = total_score
                best_solution = {
                    "assignment": candidate_solution,
                    "outcomes": decision_outcomes,
                    "stakeholder_utilities": stakeholder_utilities,
                    "weighted_stakeholder_utility": weighted_stakeholder_utility,
                    "ethical_compliance": ethical_compliance,
                    "principle_scores": self.latest_principle_scores.copy(),
                    "violations": violations.copy(),
                    "total_score": total_score
                }

            self.optimization_history.append({
                "iteration": iteration,
                "best_score": best_score,
                "ethical_compliance": ethical_compliance,
                "total_utility": weighted_stakeholder_utility
            })

        if best_solution is None:
            raise ValueError("The assignment procedure did not produce a candidate.")

        return best_solution

    def _generate_candidate_solution(
        self,
        containers: List[Dict[str, Any]],
        locations: List[Dict[str, Any]]
    ) -> Dict[int, int]:
        """Assign containers greedily while tracking each location's remaining power."""
        assignment = {}
        available_locations = {location["id"]: location.copy() for location in locations}
        sorted_containers = sorted(
            containers,
            key=lambda container: (
                container.get("urgency", 0.5) * container.get("cargo_value", 100)
            ),
            reverse=True
        )

        for container in sorted_containers:
            best_location = None
            best_local_score = -np.inf
            requested_temperature = container["temp_req"]
            tolerance = container.get("temp_tolerance", 0.0)
            permitted_min = requested_temperature - tolerance
            permitted_max = requested_temperature + tolerance

            for location_id, location in available_locations.items():
                remaining_power = location["power_cap"] - location["power_usage"]
                temperature_overlap = (
                    max(location["temp_min"], permitted_min)
                    <= min(location["temp_max"], permitted_max)
                )
                if remaining_power < container["energy"] or not temperature_overlap:
                    continue

                # This term rewards available headroom, while the other terms
                # reward temperature feasibility and the two modeled location attributes.
                headroom_ratio = remaining_power / location["power_cap"]
                ethical_bonus = 1.0
                if location.get("backup_power", False):
                    ethical_bonus += 0.1
                if location.get("emission_rating", 5) <= 3:
                    ethical_bonus += 0.15

                local_score = (
                    0.4 * headroom_ratio
                    + 0.3
                    + 0.3 * ethical_bonus
                )
                if local_score > best_local_score:
                    best_local_score = local_score
                    best_location = location_id

            if best_location is not None:
                assignment[container["id"]] = best_location
                available_locations[best_location]["power_usage"] += container["energy"]

        return assignment

    def _calculate_outcomes(
        self,
        assignment: Dict[int, int],
        containers: List[Dict[str, Any]],
        locations: List[Dict[str, Any]]
    ) -> Dict[str, float]:
        """Calculate physical totals and normalized benefit indicators."""
        containers_by_id = {container["id"]: container for container in containers}
        locations_by_id = {location["id"]: location for location in locations}
        location_usage = {location["id"]: 0 for location in locations}

        total_revenue = 0.0
        total_cost = 0.0
        total_emissions = 0.0
        safety_score = 1.0
        community_impact = 0.0
        assigned_energy = 0.0

        for container_id, location_id in assignment.items():
            container = containers_by_id[container_id]
            location = locations_by_id[location_id]
            energy = container["energy"]
            cost = location.get("base_cost", 10) * energy / 10

            total_cost += cost
            total_revenue += 1.2 * cost
            total_emissions += energy * location.get("emission_factor", 0.1)
            assigned_energy += energy
            location_usage[location_id] += 1

            if not location.get("backup_power", False):
                safety_score *= 0.95
            community_impact -= location.get("noise_level", 3) * 0.1

        usage_values = np.array(list(location_usage.values()), dtype=float)
        mean_usage = float(np.mean(usage_values)) if len(usage_values) else 0.0
        if mean_usage > 0:
            fairness_score = max(
                0.0, 1.0 - float(np.std(usage_values)) / mean_usage
            )
        else:
            fairness_score = 1.0

        total_capacity = sum(location["power_cap"] for location in locations)
        efficiency = assigned_energy / total_capacity if total_capacity > 0 else 0.0
        assignment_rate = len(assignment) / len(containers) if containers else 0.0
        emissions_benefit = max(0.0, 1.0 - total_emissions / 100.0)
        community_welfare = float(np.clip(1.0 + community_impact, 0.0, 1.0))
        transparency_score = 1.0

        outcomes = {
            "revenue": float(np.clip(total_revenue / 1000.0, 0.0, 1.0)),
            "cost": float(np.clip(1.0 - total_cost / 1000.0, 0.0, 1.0)),
            "profit": total_revenue - total_cost,
            "emissions": total_emissions,
            "sustainability": emissions_benefit,
            "environment": emissions_benefit,
            "safety_score": safety_score,
            "safety": safety_score,
            "fairness_score": fairness_score,
            "community_impact": community_impact,
            "community_welfare": community_welfare,
            "efficiency": float(np.clip(efficiency, 0.0, 1.0)),
            "utilization_rate": float(np.clip(efficiency, 0.0, 1.0)),
            "reliability": safety_score,
            "service_quality": assignment_rate,
            "employment": assignment_rate,
            "noise_reduction": community_welfare,
            "compliance": float(np.clip(1.0 - total_emissions / 100.0, 0.0, 1.0)),
            "enforcement": fairness_score,
            "throughput": assignment_rate,
            "reputation": transparency_score,
            "transparency_score": transparency_score,
            "assigned_containers": len(assignment),
            "total_containers": len(containers),
            "assigned_energy": assigned_energy,
            "total_capacity": total_capacity
        }
        return outcomes

    def generate_ethical_report(self, solution: Dict[str, Any]) -> Dict[str, Any]:
        """Summarize the selected candidate and the model's reporting indicators."""
        outcomes = solution["outcomes"]
        report = {
            "executive_summary": {
                "total_score": solution["total_score"],
                "ethical_compliance": solution["ethical_compliance"],
                "weighted_stakeholder_utility": solution["weighted_stakeholder_utility"],
                "violations_count": len(solution["violations"]),
                "profit": outcomes["profit"],
                "emissions": outcomes["emissions"],
                "safety_score": outcomes["safety_score"]
            },
            "stakeholder_analysis": {},
            "ethical_compliance": {},
            "recommendations": [],
            "transparency_metrics": {
                "decision_rationale_available": True,
                "stakeholder_impacts_disclosed": True,
                "ethical_principles_disclosed": True,
                "audit_trail_recorded": False,
                "transparency_score": outcomes["transparency_score"]
            }
        }

        for stakeholder in self.stakeholders:
            utility = solution["stakeholder_utilities"].get(stakeholder.id, 0.0)
            report["stakeholder_analysis"][stakeholder.name] = {
                "utility": utility,
                "satisfaction": float(np.clip(utility, 0.0, 1.0)),
                "priority": stakeholder.priority,
                "veto_power": stakeholder.veto_power
            }

        for principle in self.ethical_principles:
            report["ethical_compliance"][principle.name] = {
                "compliance_score": solution["principle_scores"][principle.name],
                "weight": principle.weight,
                "marked_as_hard_constraint": principle.hard_constraint,
                "type": principle.principle_type
            }

        if solution["ethical_compliance"] < 0.8:
            report["recommendations"].append(
                "Review the weighted ethical indicators and the modeled thresholds."
            )
        if outcomes["emissions"] > 50:
            report["recommendations"].append(
                "Review location choices to reduce modeled emissions."
            )
        if outcomes["safety_score"] < 0.95:
            report["recommendations"].append(
                "Review assignments to increase the use of backup-power locations."
            )

        return report


print("Value-aligned optimization class defined.")

## Example data

The following instance has eight reefer containers and four storage locations. Energy values and power capacities are expressed in kW for this illustrative model. Temperature requirements and location ranges use the same temperature scale; a container's tolerance defines its permitted temperature interval. Cargo value is recorded in thousands of dollars and contributes to the greedy processing priority together with urgency.

In [ ]:
containers = [
    {"id": 1, "energy": 45, "temp_req": -18, "temp_tolerance": 2.0, "dwell_time": 96,
     "cargo_type": "Pharmaceutical", "cargo_value": 500, "urgency": 0.9, "flexibility": 0.3},
    {"id": 2, "energy": 65, "temp_req": 3, "temp_tolerance": 1.5, "dwell_time": 72,
     "cargo_type": "Fresh Produce", "cargo_value": 80, "urgency": 0.7, "flexibility": 0.6},
    {"id": 3, "energy": 50, "temp_req": -20, "temp_tolerance": 1.0, "dwell_time": 120,
     "cargo_type": "Frozen Seafood", "cargo_value": 120, "urgency": 0.8, "flexibility": 0.2},
    {"id": 4, "energy": 35, "temp_req": 5, "temp_tolerance": 2.5, "dwell_time": 48,
     "cargo_type": "Dairy Products", "cargo_value": 60, "urgency": 0.4, "flexibility": 0.8},
    {"id": 5, "energy": 55, "temp_req": -15, "temp_tolerance": 1.8, "dwell_time": 84,
     "cargo_type": "Chemicals", "cargo_value": 200, "urgency": 0.9, "flexibility": 0.4},
    {"id": 6, "energy": 40, "temp_req": 2, "temp_tolerance": 2.0, "dwell_time": 60,
     "cargo_type": "Beverages", "cargo_value": 40, "urgency": 0.3, "flexibility": 0.7},
    {"id": 7, "energy": 60, "temp_req": -22, "temp_tolerance": 0.8, "dwell_time": 108,
     "cargo_type": "Vaccines", "cargo_value": 800, "urgency": 1.0, "flexibility": 0.1},
    {"id": 8, "energy": 48, "temp_req": -12, "temp_tolerance": 2.2, "dwell_time": 96,
     "cargo_type": "Ice Cream", "cargo_value": 90, "urgency": 0.5, "flexibility": 0.6}
]

locations = [
    {"id": 1, "power_cap": 120, "temp_min": -30, "temp_max": -5, "base_cost": 15,
     "backup_power": True, "location_type": "Ultra-Cold", "emission_factor": 0.05,
     "emission_rating": 2, "noise_level": 2, "power_usage": 0},
    {"id": 2, "power_cap": 100, "temp_min": -25, "temp_max": 8, "base_cost": 12,
     "backup_power": True, "location_type": "Cold", "emission_factor": 0.08,
     "emission_rating": 3, "noise_level": 3, "power_usage": 0},
    {"id": 3, "power_cap": 85, "temp_min": -20, "temp_max": 10, "base_cost": 10,
     "backup_power": False, "location_type": "Cool", "emission_factor": 0.12,
     "emission_rating": 4, "noise_level": 4, "power_usage": 0},
    {"id": 4, "power_cap": 90, "temp_min": -15, "temp_max": 12, "base_cost": 11,
     "backup_power": True, "location_type": "General", "emission_factor": 0.10,
     "emission_rating": 3, "noise_level": 3, "power_usage": 0}
]

optimizer = ValueAlignedOptimization(
    stakeholders, ethical_principles, utility_functions
)

print("Value-aligned optimization system initialized:")
print(f"  Containers: {len(containers)}")
print(f"  Locations: {len(locations)}")
print(f"  Stakeholders: {len(stakeholders)}")
print(f"  Ethical principles: {len(ethical_principles)}")
print(f"  Greedy constructions: {optimizer.optimization_iterations}")

print("\nContainer data:")
for container in containers:
    print(
        f"  C{container['id']}: {container['cargo_type']}, "
        f"cargo value=${container['cargo_value']}K, "
        f"urgency={container['urgency']:.1f}"
    )

print("\nLocation data:")
for location in locations:
    backup_status = "Yes" if location["backup_power"] else "No"
    print(
        f"  L{location['id']}: {location['location_type']}, "
        f"capacity={location['power_cap']} kW, backup power={backup_status}, "
        f"emission rating={location['emission_rating']}"
    )

## Run the assignment procedure

For each greedy construction, the optimizer orders containers by urgency multiplied by cargo value, checks power and temperature feasibility, and selects the feasible location with the largest local score. It then calculates system outcomes, stakeholder utilities, and weighted ethical compliance. The returned candidate is the one with the highest aggregate score after the specified penalty for each detected violation.

In [ ]:
print("\n" + "=" * 60)
print("VALUE-ALIGNED OPTIMIZATION EXECUTION")
print("=" * 60)

start_time = time.time()
solution = optimizer.optimize_assignment(containers, locations)
execution_time = time.time() - start_time
ethical_report = optimizer.generate_ethical_report(solution)

print("\nOptimization complete.")
print(f"  Execution time: {execution_time:.3f} seconds")
print(f"  Candidate score: {solution['total_score']:.3f}")
print(f"  Weighted ethical compliance: {solution['ethical_compliance']:.3f}")
print(f"  Detected violations: {len(solution['violations'])}")

if solution["violations"]:
    print("\nDetected violations:")
    for violation in solution["violations"]:
        print(f"  - {violation}")
else:
    print("\nNo modeled ethical thresholds were violated.")

print("\nAssignment results:")
for container_id, location_id in solution["assignment"].items():
    container = next(c for c in containers if c["id"] == container_id)
    location = next(location for location in locations if location["id"] == location_id)
    print(
        f"  Container {container_id} ({container['cargo_type']}) -> "
        f"Location {location_id} ({location['location_type']})"
    )

outcomes = solution["outcomes"]
print("\nSystem outcomes:")
print(f"  Modeled profit: ${outcomes['profit']:.2f}")
print(f"  Modeled emissions: {outcomes['emissions']:.2f} units")
print(f"  Safety indicator: {outcomes['safety_score']:.3f}")
print(f"  Location-use fairness indicator: {outcomes['fairness_score']:.3f}")
print(f"  Assigned energy / total capacity: {outcomes['efficiency']:.3f}")
print(
    f"  Containers assigned: {outcomes['assigned_containers']}/"
    f"{outcomes['total_containers']}"
)

print("\nStakeholder utility and modeled satisfaction:")
for stakeholder in stakeholders:
    utility = solution["stakeholder_utilities"].get(stakeholder.id, 0.0)
    satisfaction = ethical_report["stakeholder_analysis"][stakeholder.name]["satisfaction"]
    print(
        f"  {stakeholder.name}: utility={utility:.3f}, "
        f"modeled satisfaction={satisfaction:.3f}"
    )

summary = ethical_report["executive_summary"]
print("\nEthical report summary:")
print(f"  Candidate score: {summary['total_score']:.3f}")
print(f"  Weighted ethical compliance: {summary['ethical_compliance']:.3f}")
print(f"  Priority-weighted stakeholder utility: {summary['weighted_stakeholder_utility']:.3f}")
print(f"  Detected violations: {summary['violations_count']}")
print(f"  Modeled profit: ${summary['profit']:.2f}")
print(f"  Modeled emissions: {summary['emissions']:.2f} units")
print(f"  Safety indicator: {summary['safety_score']:.3f}")

if ethical_report["recommendations"]:
    print("\nModel-based review prompts:")
    for number, recommendation in enumerate(ethical_report["recommendations"], start=1):
        print(f"  {number}. {recommendation}")
else:
    print("\nNo review prompts were triggered by the modeled thresholds.")

# Verify that the reported assignment obeys capacity and temperature feasibility.
assigned_energy_by_location = defaultdict(float)
for container_id, location_id in solution["assignment"].items():
    container = next(c for c in containers if c["id"] == container_id)
    location = next(location for location in locations if location["id"] == location_id)
    assigned_energy_by_location[location_id] += container["energy"]
    permitted_min = container["temp_req"] - container["temp_tolerance"]
    permitted_max = container["temp_req"] + container["temp_tolerance"]
    assert max(location["temp_min"], permitted_min) <= min(location["temp_max"], permitted_max)

for location in locations:
    assert assigned_energy_by_location[location["id"]] <= location["power_cap"]

assert math.isclose(
    sum(assigned_energy_by_location.values()),
    outcomes["assigned_energy"],
    rel_tol=1e-9,
    abs_tol=1e-9
)
assert math.isclose(
    solution["total_score"],
    optimizer.optimization_history[-1]["best_score"],
    rel_tol=1e-9,
    abs_tol=1e-9
)
print("\nAssignment capacity, temperature, and score checks passed.")

## Visual inspection of model indicators

The dashboard summarizes the selected assignment and the recorded optimization history. Values are indicators defined by this example's model; they should not be interpreted as independently measured social, environmental, or regulatory outcomes. The utility and transparency panels expose the model's inputs and reporting conventions rather than empirical stakeholder feedback or a completed external audit.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle(
    "Value-Aligned Reefer Assignment: Model Indicators",
    fontsize=16,
    fontweight="bold"
)

# Stakeholder satisfaction is a bounded representation of modeled utility.
ax = axes[0, 0]
stakeholder_names = [stakeholder.name for stakeholder in stakeholders]
satisfactions = [
    ethical_report["stakeholder_analysis"][name]["satisfaction"]
    for name in stakeholder_names
]
priorities = [stakeholder.priority for stakeholder in stakeholders]
ax.scatter(priorities, satisfactions, s=180, alpha=0.7, color="teal")
for name, priority, satisfaction in zip(stakeholder_names, priorities, satisfactions):
    ax.annotate(
        name.split()[0], (priority, satisfaction),
        xytext=(5, 5), textcoords="offset points", fontsize=8
    )
ax.set_xlabel("Stakeholder priority (weight)")
ax.set_ylabel("Modeled satisfaction (0-1)")
ax.set_title("Modeled Satisfaction and Priority")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# A polar plot displays one compliance score for each named principle.
ax = fig.add_subplot(3, 3, 2, projection="polar")
principle_names = [principle.name for principle in ethical_principles]
compliance_scores = [
    solution["principle_scores"][name] for name in principle_names
]
angles = np.linspace(0, 2 * np.pi, len(principle_names), endpoint=False)
closed_angles = np.concatenate([angles, angles[:1]])
closed_scores = compliance_scores + compliance_scores[:1]
ax.plot(closed_angles, closed_scores, marker="o", linewidth=2, label="Compliance")
ax.fill(closed_angles, closed_scores, alpha=0.2)
ax.set_xticks(angles)
ax.set_xticklabels([name.split()[0] for name in principle_names], fontsize=8)
ax.set_ylim(0, 1)
ax.set_title("Principle-Level Compliance", pad=18)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))

# The line shows the best score recorded through each repeated construction.
ax = axes[0, 2]
iterations = [entry["iteration"] for entry in optimizer.optimization_history]
best_scores = [entry["best_score"] for entry in optimizer.optimization_history]
ax.plot(iterations, best_scores, color="navy", linewidth=2, label="Best score so far")
ax.set_xlabel("Greedy construction iteration")
ax.set_ylabel("Candidate score")
ax.set_title("Recorded Best Score")
ax.grid(True, alpha=0.3)
ax.legend()

# Compare three economic and three ethical indicators on a shared 0-100 scale.
ax = axes[1, 0]
comparison_labels = ["Efficiency", "Safety", "Fairness"]
comparison_values = [
    outcomes["efficiency"] * 100,
    outcomes["safety_score"] * 100,
    outcomes["fairness_score"] * 100
]
bars = ax.bar(comparison_labels, comparison_values, color=["seagreen", "royalblue", "darkorange"])
ax.set_ylabel("Modeled indicator (0-100)")
ax.set_title("Operational and Ethical Indicators")
ax.set_ylim(0, 105)
ax.grid(axis="y", alpha=0.3)
for bar, value in zip(bars, comparison_values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 1, f"{value:.1f}", ha="center")

# Emissions are grouped by the location type used in the selected assignment.
ax = axes[1, 1]
emissions_by_type = defaultdict(float)
for container_id, location_id in solution["assignment"].items():
    container = next(c for c in containers if c["id"] == container_id)
    location = next(location for location in locations if location["id"] == location_id)
    emissions_by_type[location["location_type"]] += (
        container["energy"] * location["emission_factor"]
    )
emission_labels = list(emissions_by_type)
emission_values = [emissions_by_type[label] for label in emission_labels]
if sum(emission_values) > 0:
    ax.pie(
        emission_values,
        labels=emission_labels,
        autopct="%1.1f%%",
        startangle=90
    )
else:
    ax.text(0.5, 0.5, "No assigned-container emissions", ha="center", va="center")
ax.set_title("Modeled Emissions by Location Type")

# Show each stakeholder's specified utility weights, not realized utility components.
ax = axes[1, 2]
utility_labels = []
utility_weights = []
for stakeholder in stakeholders:
    for objective, weight in stakeholder.utilities.items():
        utility_labels.append(f"{stakeholder.name.split()[0]}: {objective}")
        utility_weights.append(weight)
positions = np.arange(len(utility_labels))
ax.barh(positions, utility_weights, color="slateblue", alpha=0.8)
ax.set_yticks(positions)
ax.set_yticklabels(utility_labels, fontsize=7)
ax.set_xlabel("Specified utility weight")
ax.set_title("Stakeholder Objective Weights")
ax.grid(axis="x", alpha=0.3)

# Each bar corresponds to one ethical principle, including transparency.
ax = axes[2, 0]
principle_colors = ["forestgreen", "royalblue", "firebrick", "darkorange", "purple"]
principle_bars = ax.bar(
    [name.split()[0] for name in principle_names],
    [score * 100 for score in compliance_scores],
    color=principle_colors,
    alpha=0.8
)
ax.set_xlabel("Ethical principle")
ax.set_ylabel("Modeled compliance (%)")
ax.set_title("Principle Threshold Indicators")
ax.set_ylim(0, 105)
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.3)

# Boolean reporting fields describe available information, not verified audits.
ax = axes[2, 1]
transparency_items = ethical_report["transparency_metrics"]
transparency_keys = [
    "decision_rationale_available",
    "stakeholder_impacts_disclosed",
    "ethical_principles_disclosed",
    "audit_trail_recorded"
]
transparency_labels = ["Rationale", "Impacts", "Principles", "Audit trail"]
transparency_values = [float(transparency_items[key]) for key in transparency_keys]
ax.bar(transparency_labels, transparency_values, color="purple", alpha=0.75)
ax.set_ylabel("Modeled availability (0 or 1)")
ax.set_title("Reporting and Audit Fields")
ax.set_ylim(0, 1.1)
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.3)

# Environmental benefit is shown as a normalized indicator, not an emissions reduction claim.
ax = axes[2, 2]
performance_labels = ["Ethical compliance", "Mean satisfaction", "Efficiency", "Emissions benefit"]
mean_satisfaction = float(np.mean(satisfactions)) if satisfactions else 0.0
performance_values = [
    solution["ethical_compliance"],
    mean_satisfaction,
    outcomes["efficiency"],
    outcomes["sustainability"]
]
performance_colors = [
    "green" if value >= 0.8 else "orange" if value >= 0.6 else "red"
    for value in performance_values
]
performance_bars = ax.bar(
    performance_labels,
    [value * 100 for value in performance_values],
    color=performance_colors,
    alpha=0.8
)
for bar, value in zip(performance_bars, performance_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value * 100 + 1,
        f"{value * 100:.1f}%",
        ha="center",
        fontsize=8
    )
ax.set_ylabel("Normalized model indicator (%)")
ax.set_title("Summary of Selected Candidate")
ax.set_ylim(0, 110)
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print("\nDashboard panels:")
print("  1. Stakeholder modeled satisfaction and priority")
print("  2. Weighted ethical principle scores")
print("  3. Recorded best score during the configured iterations")
print("  4. Operational and ethical indicators")
print("  5. Modeled emissions by assigned location type")
print("  6. Stakeholder objective weights")
print("  7. Principle-level compliance indicators")
print("  8. Reporting and audit-field availability")
print("  9. Normalized indicators for the selected candidate")

print("\nSelected-candidate indicators:")
print(f"  Weighted ethical compliance: {solution['ethical_compliance']:.3f}")
print(f"  Mean modeled stakeholder satisfaction: {mean_satisfaction:.3f}")
print(f"  Modeled emissions: {outcomes['emissions']:.2f} units")
print(f"  Safety indicator: {outcomes['safety_score']:.3f}")
print(f"  Location-use fairness indicator: {outcomes['fairness_score']:.3f}")
print(
    "  Transparency indicator: "
    f"{ethical_report['transparency_metrics']['transparency_score']:.3f}"
)

## Strengths, limitations, and appropriate use

### What this framework illustrates

- Multiple stakeholder preferences can be represented by weighted utility functions and combined using stakeholder priorities.
- Ethical indicators can be evaluated alongside stakeholder utility, and detected violations can affect a candidate's score.
- A structured report can expose assignments, modeled outcomes, principle-level scores and the rationale components used by the example.

### Limitations

- The greedy assignment rule is locally informed and does not guarantee a globally best assignment. Repeating the same deterministic construction does not add search diversity.
- Stakeholder constraints, veto powers, utility-objective weights stored in `UtilityFunction.objectives`, time preferences, and principle-specific `violation_penalty` values are metadata in this implementation; they do not affect candidate selection. The optimization uses `Stakeholder.utilities`, stakeholder priorities, the global ethical weight and a fixed penalty of 50 per detected violation.
- Principles labeled as hard constraints are scored and may trigger penalties, but they do not make a candidate infeasible. The framework therefore does not guarantee regulatory compliance or ethical acceptability.
- The utility and impact mappings are illustrative proxies. They require empirical calibration, stakeholder validation and careful treatment of conflicting objectives before use in a real terminal.
- The reported transparency score reflects the example's reporting design; it is not evidence of an independent audit, explainability validation or stakeholder trust.

### Appropriate use

Use this example to study how a greedy assignment procedure can be paired with stakeholder utility, weighted ethical indicators, violation penalties and transparent reporting. Any operational deployment would require validated data, domain-specific constraints, meaningful measurement units, stakeholder governance and independent review.